# Phase 0: FreeTwinEV Dataset Inspection and Assessment

**Course:** Advanced AI Techniques (AAI)
**Project:** AI-Enabled Micro-channel Battery Thermal Management System for Electric Vehicles (KJS-CES-02)
**Candidate Dataset:** FreeTwinEV P45B 1s4p Battery Module Measurement

This notebook provides the technical inspection and diagnostics for the FreeTwinEV dataset across:
- Provenance and experimental protocol
- Physical structure of the 14 measurement runs
- Temporal continuity and sampling stability
- Temperature sensor mapping (ambient, coolant, cold plate, cell sensors)
- Electrical and cooling operating conditions
- Thermal continuity vs. the rejected dataset
- Future forecasting feasibility and train/test holdout strategy

In [ ]:
import os
import sys
import glob
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

print('Project root:', project_root)
all_data_dir = project_root / 'data' / 'candidate_freetwinev' / '1s4p Battery Module Measurement' / 'AllData'
print('Candidate data exists:', all_data_dir.exists())

## 1. Experimental Runs Overview Table

Overview of all 14 runs extracted from `results/freetwinev/runs_overview.csv`.

In [ ]:
df_runs = pd.read_csv(project_root / 'results' / 'freetwinev' / 'runs_overview.csv')
display_cols = ['Exp_ID', 'Rows', 'Duration_min', 'Sampling_Median_s', 'I_min_A', 'Flow_mean_Lmin', 'Tinlet_mean_C', 'Tcell_max_peak_C', 'Tcell_spread_max_C', 'dT_step_mean_C', 'Autocorr_Lag1_5s', 'Autocorr_Lag12_60s']
df_runs[display_cols]

## 2. Experimental Setup and Channel Mapping

Let us inspect the columns and units of a representative run (ID01).

In [ ]:
sample_csv = list(all_data_dir.glob('ID01*/*.csv'))[0]
df_sample = pd.read_csv(sample_csv)
units = df_sample.iloc[0].to_dict()
print('Total columns:', len(df_sample.columns))
print('\nChannel Units:')
for col, unit in units.items():
    print(f'  - {col}: {unit}')

## 3. Physical Trajectory of a Representative Run (ID13)

ID13 represents high-rate discharge (-120 A, ~6.7C), 15°C coolant, 1.5 L/min flow.

In [ ]:
id13_csv = list(all_data_dir.glob('ID13*/*.csv'))[0]
df13 = pd.read_csv(id13_csv).iloc[1:].apply(pd.to_numeric)
t_min = df13['timesamp'] / 60.0

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(t_min, df13['current'], color='crimson', label='Current [A]')
axes[0].set_ylabel('Current [A]')
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].plot(t_min, df13['liquid_flow'], color='darkorange', label='Flow [L/min]')
axes[1].plot(t_min, df13['temp_inlet'] - 273.15, color='blue', label='Coolant Inlet [°C]')
axes[1].plot(t_min, df13['temp_outlet'] - 273.15, color='cyan', label='Coolant Outlet [°C]')
axes[1].set_ylabel('Cooling Conditions')
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle='--', alpha=0.5)

cell_cols = [c for c in df13.columns if 'cell' in c]
for c in cell_cols:
    axes[2].plot(t_min, df13[c] - 273.15, alpha=0.4, linewidth=1)
axes[2].plot(t_min, np.max(df13[cell_cols].values - 273.15, axis=1), color='black', linewidth=2, label='Max Cell Temp [°C]')
axes[2].axhline(70, color='red', linestyle='--', label='70°C Cut-off')
axes[2].set_ylabel('Temperature [°C]')
axes[2].set_xlabel('Elapsed Time [min]')
axes[2].legend(loc='upper right')
axes[2].grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Continuity Comparison: FreeTwinEV vs Rejected Dataset

In [ ]:
df_old = pd.read_csv(project_root / 'EV_Battery_Charging_TR_Dataset_with_Notes (1).csv')
old_deltas = np.abs(np.diff(df_old['MaxTemp_C'].values))
print(f'Rejected dataset 1-minute temperature jump: Mean = {np.mean(old_deltas):.2f}°C, Max = {np.max(old_deltas):.2f}°C')

ft_deltas = np.abs(np.diff(np.max(df13[cell_cols].values - 273.15, axis=1)))
print(f'FreeTwinEV (ID13) 5-second temperature change: Mean = {np.mean(ft_deltas):.4f}°C, Max = {np.max(ft_deltas):.2f}°C')